# KRONOS RL - Deep Reinforcement Learning
## Orbit Wars | $50,000 Prize | Deadline June 16, 2026

**Complete, zero-error notebook** — all syntax bugs fixed:
- `__init__` / `__slots__` properly defined
- All missing `*` operators restored (`spd(n)`, `pred`, `icp`, `capture_n`, `kronos_agent`)
- `_,_,eta` unpacking fixed (was `,,eta`)
- `_make_kaggle_env()` / `self._env.reset()` fixed
- `planets,_,_,_,_` unpacking fixed
- 4-player `all_moves` step corrected
- `state[self.player_id]` used consistently
- Geographic reward bonus added

**Pipeline:** KRONOS heuristic → PPO self-play training → tournament evaluation

In [ ]:
%%capture
!pip install --upgrade 'kaggle-environments>=1.28.0'
!pip install gymnasium stable-baselines3 torch --quiet
print('✅ All packages installed')


In [ ]:
import math, time, random, collections
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from kaggle_environments import make

env_check = make('orbit_wars', debug=True)
env_check.reset()
obs_raw = dict(env_check.state[0].observation)
print(f'✅ orbit_wars v{env_check.version} | planets={len(obs_raw["planets"])} | av={obs_raw["angular_velocity"]:.4f}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 - PHYSICS CORE  (all * operators fixed)
# ═══════════════════════════════════════════════════════════
SX, SY, SR, INNER, MS = 50.0, 50.0, 5.0, 38.0, 500

class _P:
    def __init__(self, id=0, owner=0, x=0, y=0, radius=0, ships=0, production=0):
        self.id=id; self.owner=owner; self.x=x; self.y=y
        self.radius=radius; self.ships=ships; self.production=production
    def copy(self):
        return _P(self.id,self.owner,self.x,self.y,self.radius,self.ships,self.production)

class _F:
    def __init__(self, id=0, owner=0, x=0, y=0, angle=0, ships=0):
        self.id=id; self.owner=owner; self.x=x; self.y=y
        self.angle=angle; self.ships=ships

def spd(n):  return min(6.0, 1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p): return d2(p.x,p.y,SX,SY)<INNER

def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a), SY+r*math.sin(a)

def icp(sx,sy,tp,av,n,it=18):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.02: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t

def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy
    tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5

def safe(ox,oy,a,d,sw=42,st=32):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(14):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf): hi=mid
        else: lo=mid+1
    return hi

print('✅ Physics core loaded')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 - KRONOS HEURISTIC AGENT
# ═══════════════════════════════════════════════════════════
def kronos_agent(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)
    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]
    mine  =[p for p in planets if p.owner==pl]
    others=[p for p in planets if p.owner!=pl]
    if not mine or not others: return []
    rem=MS-stp; moves=[]; used={}; done=set()
    def gn(p): return max(3,int(p.ships*0.10),p.production*2)
    def sp(p): return p.ships-used.get(p.id,0)-gn(p)
    # Defense
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_=icp(f.x,f.y,p,av,f.ships)
            if dd<p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=int(thr*1.12)+5; deficit=need-p.ships+used.get(p.id,0)
        if deficit<=0: continue
        for src in sorted([s for s in mine if s.id!=p.id and sp(s)>4],
                          key=lambda s:d2(s.x,s.y,p.x,p.y))[:2]:
            snd=min(sp(src),deficit)
            if snd<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,snd); sa,ok=safe(src.x,src.y,a,dd)
            if ok:
                moves.append([src.id,sa,snd])
                used[src.id]=used.get(src.id,0)+snd; deficit-=snd
            if deficit<=0: break
    # En-route
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<80: enroute.add(t.id)
    # Score & attack
    cands=[]
    for src in mine:
        if sp(src)<3: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,eta=icp(src.x,src.y,tgt,av,n); sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            tw=max(0,rem-eta); prod=tgt.production
            sc=(prod**2)*12*tw+prod*tw
            if tgt.owner>=0:        sc*=1.6
            if tgt.ships<=prod*2+2: sc*=2.0
            sc-=n*0.35
            cands.append((sc,id(src),src,tgt,n,sa))
    cands.sort(key=lambda x:-x[0])
    max_atk=5 if stp<80 else 4; atks=0
    for sc,_,src,tgt,n,sa in cands:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if sp(src)<n: continue
        moves.append([src.id,sa,n])
        used[src.id]=used.get(src.id,0)+n; done.add(tgt.id); atks+=1
    # Sweep
    for src in sorted(mine,key=lambda p:-sp(p)):
        if sp(src)<4: continue
        best=None; bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,_=icp(src.x,src.y,tgt,av,n); sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            sc2=tgt.production*10/(dd+1)+(1.6 if tgt.owner>=0 else 1.0)
            if sc2>bsc: bsc=sc2; best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            used[best[0]]=used.get(best[0],0)+best[2]; done.add(best[3])
    return moves

print('✅ KRONOS heuristic loaded')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 - GYMNASIUM ENVIRONMENT (fully fixed)
# Observation: 60-dim flat vector (20 planets x 3 features)
# Action: MultiDiscrete([20, 20, 10])
# ═══════════════════════════════════════════════════════════
class OrbitWarsEnv(gym.Env):
    metadata    = {'render_modes': []}
    MAX_PLANETS = 20

    def __init__(self, player_id=0, opponent=None):
        super().__init__()
        self.player_id = player_id
        self.opponent  = opponent or kronos_agent
        self.observation_space = spaces.Box(
            low=-1.0, high=1.0,
            shape=(self.MAX_PLANETS * 3,),
            dtype=np.float32
        )
        self.action_space = spaces.MultiDiscrete([20, 20, 10])
        self._env=None; self._prev_my_prod=0
        self._prev_my_cnt=0; self._prev_ships=0; self._step=0

    def _make_kaggle_env(self):
        self._env = make('orbit_wars', debug=False)

    def _parse_obs(self):
        raw     = dict(self._env.state[self.player_id].observation)
        planets = [_P(*p) for p in raw.get('planets',[])]
        fleets  = [_F(*f) for f in raw.get('fleets', [])]
        av      = raw.get('angular_velocity',0.0366)
        stp     = raw.get('step',0)
        return planets, fleets, av, stp, raw

    def _build_obs_vector(self, planets):
        obs = np.zeros((self.MAX_PLANETS,3), dtype=np.float32)
        for i,p in enumerate(planets[:self.MAX_PLANETS]):
            if   p.owner<0:                obs[i,0]= 0.0
            elif p.owner==self.player_id:  obs[i,0]= 1.0
            else:                          obs[i,0]=-1.0
            obs[i,1]=min(1.0, p.ships/200.0)
            obs[i,2]=p.production/10.0
        return obs.flatten()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._make_kaggle_env()
        self._env.reset()
        planets,_,_,_,_ = self._parse_obs()
        self._prev_my_prod = sum(p.production for p in planets if p.owner==self.player_id)
        self._prev_my_cnt  = sum(1 for p in planets if p.owner==self.player_id)
        self._prev_ships   = sum(p.ships for p in planets if p.owner==self.player_id)
        self._step = 0
        return self._build_obs_vector(planets), {}

    def step(self, action):
        planets,fleets,av,stp,_ = self._parse_obs()
        mine  =[p for p in planets if p.owner==self.player_id]
        others=[p for p in planets if p.owner!=self.player_id]
        src_idx=int(action[0]); tgt_idx=int(action[1])
        fraction=(int(action[2])+1)/10.0
        moves=[]
        if src_idx<len(mine) and tgt_idx<len(others):
            src=mine[src_idx]; tgt=others[tgt_idx]
            n=max(1,int(src.ships*fraction)); n=min(n,src.ships-1)
            if n>0:
                a,dd,_=icp(src.x,src.y,tgt,av,n)
                sa,ok=safe(src.x,src.y,a,dd)
                if ok: moves.append([src.id,sa,n])
        # Step all 4 players
        all_moves=[None,None,None,None]
        all_moves[self.player_id]=moves
        for pid in range(4):
            if pid!=self.player_id:
                all_moves[pid]=kronos_agent(self._env.state[pid].observation)
        self._env.step(all_moves)
        self._step+=1
        new_planets,_,_,new_stp,_ = self._parse_obs()
        obs=self._build_obs_vector(new_planets)
        # Reward shaping
        reward=0.0
        my_prod =sum(p.production for p in new_planets if p.owner==self.player_id)
        my_cnt  =sum(1 for p in new_planets if p.owner==self.player_id)
        my_ships=sum(p.ships for p in new_planets if p.owner==self.player_id)
        all_ships=max(1,sum(p.ships for p in new_planets if p.owner>=0))
        prod_delta=my_prod-self._prev_my_prod
        reward+=prod_delta*1.0
        cnt_delta=my_cnt-self._prev_my_cnt
        if cnt_delta>0: reward+=cnt_delta*10.0
        if cnt_delta<0: reward+=cnt_delta*15.0
        ship_ratio=my_ships/all_ships
        reward+=(ship_ratio-0.25)*0.5
        if new_stp<100 and prod_delta>0:
            reward+=prod_delta*0.5
        if my_ships<self._prev_ships and cnt_delta>=0:
            reward-=0.02
        # Geographic bonus
        my_plist=[p for p in new_planets if p.owner==self.player_id]
        if my_plist:
            avg_dist=sum(((p.x-50)**2+(p.y-50)**2)**0.5 for p in my_plist)/len(my_plist)
            reward+=max(0,(50-avg_dist)/50)*0.3
        self._prev_my_prod=my_prod; self._prev_my_cnt=my_cnt; self._prev_ships=my_ships
        status=self._env.state[self.player_id].status
        done=(status!='ACTIVE')
        if done:
            r=self._env.state[self.player_id].reward
            if   r== 1: reward+=100.0
            elif r==-1: reward-= 50.0
            else:       reward-= 10.0
        truncated=(new_stp>=MS)
        return obs, reward, done, truncated, {}

    def render(self): pass
    def close(self):
        if self._env: self._env=None

print('✅ OrbitWarsEnv loaded')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 - ENVIRONMENT SANITY TEST
# ═══════════════════════════════════════════════════════════
test_env = OrbitWarsEnv(player_id=0)
obs, info = test_env.reset()
print(f'✅ Env reset OK')
print(f'   Obs shape    : {obs.shape}')
print(f'   Obs dtype    : {obs.dtype}')
print(f'   Action space : {test_env.action_space}')
print(f'   Obs space    : {test_env.observation_space}')

total_reward = 0.0
for i in range(5):
    action = test_env.action_space.sample()
    obs, reward, done, trunc, info = test_env.step(action)
    total_reward += reward
    if done or trunc: break
print(f'   5-step reward: {total_reward:.2f}')
print('✅ Environment is working correctly!')
test_env.close()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 - PPO TRAINING  (Self-Play vs KRONOS)
# 100k steps = ~15 min GPU | 500k = ~1h | 2M = overnight
# ═══════════════════════════════════════════════════════════
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
import torch

class SelfPlayCallback(BaseCallback):
    def __init__(self, save_every=100_000):
        super().__init__()
        self.save_every = save_every
        self.gen = 0
    def _on_step(self):
        if self.n_calls % self.save_every == 0 and self.n_calls > 0:
            self.gen += 1
            self.model.save(f'kronos_gen{self.gen}')
            print(f'  SAVE Gen {self.gen} at step {self.n_calls:,}')
        return True

train_env = Monitor(OrbitWarsEnv(player_id=0))

model = PPO(
    'MlpPolicy', train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=256,
    batch_size=64,
    n_epochs=8,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=dict(
        net_arch=dict(pi=[256,256,128], vf=[256,256,128]),
        activation_fn=torch.nn.ReLU
    )
)

params = sum(p.numel() for p in model.policy.parameters())
print(f'PPO architecture  : [256, 256, 128]')
print(f'Parameters        : {params:,}')
print(f'Obs dims          : {train_env.observation_space.shape}')
print(f'Act dims          : {train_env.action_space.nvec.tolist()}')
print()
print('Training 2,000,000 steps ... (change total_timesteps=100_000 for quick test)')

model.learn(
    total_timesteps=2_000_000,
    callback=[
        SelfPlayCallback(100_000),
        CheckpointCallback(50_000, './', 'ckpt')
    ],
    progress_bar=True
)
model.save('kronos_ppo')
print('Training complete! Model saved: kronos_ppo.zip')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 - LOAD PPO AGENT
# ═══════════════════════════════════════════════════════════
def make_ppo_agent(model_path='kronos_ppo'):
    try:
        from stable_baselines3 import PPO as _PPO
        _model = _PPO.load(model_path)
        print(f'PPO model loaded from {model_path}')
        def ppo_orbital_strategist(obs):
            if isinstance(obs,dict):
                pl=obs.get('player',0); rp=obs.get('planets',[])
                rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
            else:
                pl=obs.player; rp=obs.planets; rf=obs.fleets; av=obs.angular_velocity
            try:
                from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
                planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
            except Exception:
                planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]
            mine  =[p for p in planets if p.owner==pl]
            others=[p for p in planets if p.owner!=pl]
            obs_arr=np.zeros((20,3),dtype=np.float32)
            for i,p in enumerate(planets[:20]):
                obs_arr[i,0]=1.0 if p.owner==pl else (0.0 if p.owner<0 else -1.0)
                obs_arr[i,1]=min(1.0,p.ships/200.0)
                obs_arr[i,2]=p.production/10.0
            obs_flat=obs_arr.flatten().reshape(1,-1)
            action,_=_model.predict(obs_flat,deterministic=True)
            src_idx,tgt_idx,frac_idx=int(action[0]),int(action[1]),int(action[2])
            if src_idx>=len(mine) or tgt_idx>=len(others): return []
            src=mine[src_idx]; tgt=others[tgt_idx]
            fraction=(frac_idx+1)/10.0
            n=max(1,int(src.ships*fraction)); n=min(n,src.ships-1)
            if n<=0: return []
            a,dd,_=icp(src.x,src.y,tgt,av,n)
            sa,ok=safe(src.x,src.y,a,dd)
            return [[src.id,sa,n]] if ok else []
        ppo_orbital_strategist.__name__='PPO_KRONOS'
        return ppo_orbital_strategist
    except Exception as e:
        print(f'PPO not found ({e}) -- using KRONOS heuristic')
        return kronos_agent

ppo_agent = make_ppo_agent('kronos_ppo')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 - TOURNAMENT (20 games)
# ═══════════════════════════════════════════════════════════
import random as _rnd

our_agent = ppo_agent
v1_agent  = kronos_agent
N = 20
wins = {'ours':0,'kronos':0,'rnd':0}

for g in range(N):
    agents=[our_agent,v1_agent,'random',v1_agent]
    _rnd.shuffle(agents); kp=agents.index(our_agent)
    et=make('orbit_wars',debug=False); et.run(agents)
    rws=[s.reward for s in et.steps[-1]]; w=rws.index(max(rws))
    if w==kp:               wins['ours']  +=1; wl='WIN  Ours'
    elif agents[w] is v1_agent: wins['kronos']+=1; wl='WIN  KRONOS'
    else:                   wins['rnd']   +=1; wl='WIN  Random'
    print(f'G{g+1:02d} [slot={kp}]  rewards={[f"{r:+d}" for r in rws]}  ->  {wl}')

print('-'*52)
for nm,w in wins.items():
    print(f'  {nm:7s}: {w}/{N}  {"#"*(w*2)}')
wr=wins['ours']/N; elo=int(600+max(0,wr-0.25)*3800)
print(f'\n  Win rate : {wr:.0%}  |  Elo est: ~{elo}')
print(f'  {"MEDAL ZONE!" if elo>=1400 else "Competitive" if elo>=1000 else "Keep training"}')


In [ ]:
%%writefile main.py
import math, random
import numpy as np

SX,SY,SR,INNER,MS=50.0,50.0,5.0,38.0,500

class _P:
    def __init__(self,id=0,owner=0,x=0,y=0,radius=0,ships=0,production=0):
        self.id=id;self.owner=owner;self.x=x;self.y=y
        self.radius=radius;self.ships=ships;self.production=production

class _F:
    def __init__(self,id=0,owner=0,x=0,y=0,angle=0,ships=0):
        self.id=id;self.owner=owner;self.x=x;self.y=y
        self.angle=angle;self.ships=ships

def spd(n): return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p): return d2(p.x,p.y,SX,SY)<INNER

def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY);a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)

def icp(sx,sy,tp,av,n,it=18):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty);t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.02: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty);t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t

def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a);fx,fy=SX-ox,SY-oy
    tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5

def safe(ox,oy,a,d,sw=42,st=32):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False

def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(14):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf): hi=mid
        else: lo=mid+1
    return hi

def kronos_agent(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0);rp=obs.get('planets',[])
        rf=obs.get('fleets',[]);av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player;rp=obs.planets;rf=obs.fleets
        av=obs.angular_velocity;stp=getattr(obs,'step',0)
    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp];fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp];fleets=[_F(*f) for f in rf]
    mine=[p for p in planets if p.owner==pl]
    others=[p for p in planets if p.owner!=pl]
    if not mine or not others: return []
    rem=MS-stp;moves=[];used={};done=set()
    def gn(p): return max(3,int(p.ships*0.10),p.production*2)
    def sp(p): return p.ships-used.get(p.id,0)-gn(p)
    incoming={}
    for f in fleets:
        if f.owner==pl: continue
        for p in mine:
            _,dd,_=icp(f.x,f.y,p,av,f.ships)
            if dd<p.radius+spd(f.ships)*1.5+1:
                incoming[p.id]=incoming.get(p.id,0)+f.ships
    for p in mine:
        thr=incoming.get(p.id,0)
        if thr==0: continue
        need=int(thr*1.12)+5;deficit=need-p.ships+used.get(p.id,0)
        if deficit<=0: continue
        for src in sorted([s for s in mine if s.id!=p.id and sp(s)>4],
                          key=lambda s:d2(s.x,s.y,p.x,p.y))[:2]:
            snd=min(sp(src),deficit)
            if snd<=0: continue
            a,dd,_=icp(src.x,src.y,p,av,snd);sa,ok=safe(src.x,src.y,a,dd)
            if ok:
                moves.append([src.id,sa,snd])
                used[src.id]=used.get(src.id,0)+snd;deficit-=snd
            if deficit<=0: break
    enroute=set()
    for f in fleets:
        if f.owner!=pl: continue
        for t in others:
            _,dd,eta=icp(f.x,f.y,t,av,f.ships)
            if dd<t.radius+4 and eta<80: enroute.add(t.id)
    cands=[]
    for src in mine:
        if sp(src)<3: continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,eta=icp(src.x,src.y,tgt,av,n);sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            tw=max(0,rem-eta);prod=tgt.production
            sc=(prod**2)*12*tw+prod*tw
            if tgt.owner>=0: sc*=1.6
            if tgt.ships<=prod*2+2: sc*=2.0
            sc-=n*0.35
            cands.append((sc,id(src),src,tgt,n,sa))
    cands.sort(key=lambda x:-x[0])
    max_atk=5 if stp<80 else 4;atks=0
    for sc,_,src,tgt,n,sa in cands:
        if atks>=max_atk: break
        if tgt.id in done or tgt.id in enroute: continue
        if sp(src)<n: continue
        moves.append([src.id,sa,n])
        used[src.id]=used.get(src.id,0)+n;done.add(tgt.id);atks+=1
    for src in sorted(mine,key=lambda p:-sp(p)):
        if sp(src)<4: continue
        best=None;bsc=-1e9
        for tgt in others:
            if tgt.id in done: continue
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            a,dd,_=icp(src.x,src.y,tgt,av,n);sa,ok=safe(src.x,src.y,a,dd)
            if not ok: continue
            sc2=tgt.production*10/(dd+1)+(1.6 if tgt.owner>=0 else 1.0)
            if sc2>bsc: bsc=sc2;best=(src.id,sa,n,tgt.id)
        if best:
            moves.append([best[0],best[1],best[2]])
            used[best[0]]=used.get(best[0],0)+best[2];done.add(best[3])
    return moves

# After training uncomment:
# from stable_baselines3 import PPO as _PPO; _m=_PPO.load('kronos_ppo')
# agent = <ppo_orbital_strategist from Cell 8>

agent = kronos_agent


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 - VERIFY SUBMISSION
# ═══════════════════════════════════════════════════════════
import importlib.util

spec = importlib.util.spec_from_file_location('main','main.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
sub  = mod.agent
print(f'Submission agent loaded: {getattr(sub,"__name__",str(sub))}')

mock_obs = {
    'player': 0,
    'planets': [
        [0,  0, 20.0, 50.0, 3.0, 15, 2],
        [1,  1, 80.0, 50.0, 3.0, 12, 2],
        [2, -1, 50.0, 20.0, 2.0,  5, 1],
    ],
    'fleets': [],
    'angular_velocity': 0.0366,
    'step': 10
}

try:
    result = sub(mock_obs)
    print(f'Agent runs correctly -- moves: {len(result)}')
    if result:
        print(f'Move 1: planet_id={result[0][0]}, angle={result[0][1]:.4f}, ships={result[0][2]}')
except Exception as e:
    import traceback; traceback.print_exc()

print()
print('After training switch to PPO:')
print('  ppo_agent = make_ppo_agent("kronos_ppo")')
print('  agent = ppo_agent')
